<a href="https://colab.research.google.com/github/amann45/AI_Labworks/blob/main/Lab7/lab7RNN_Encoder_Decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## RNN Encoder Decoder [Aman Kumar Ray: ACE080BCT010]

## Objective: To understand the working principle of the Recurrent Neural Network (RNN) Encoder–Decoder architecture & implement a Sequence-to-Sequence (Seq2Seq) model for machine translation.

## Theory

### Recurrent Neural Network (RNN)

A **Recurrent Neural Network (RNN)** is a neural network designed to process sequential data such as text, speech, and time-series information. Unlike feedforward neural networks, an RNN maintains a hidden state that carries information from previous time steps, allowing it to capture dependencies between sequence elements.

The hidden state at time **t** is calculated as:

\[
h_t = f(W_h h_{t-1} + W_x x_t + b)
\]

where,

- \(x_t\) = Current input
- \(h_{t-1}\) = Previous hidden state
- \(W_h\), \(W_x\) = Weight matrices
- \(b\) = Bias
- \(f\) = Activation function (usually **tanh**)

Although RNNs are effective for sequential data, they suffer from the **vanishing gradient problem**, making it difficult to learn long-term dependencies.

---

### Encoder–Decoder Architecture

The **Encoder–Decoder** architecture, also called the **Sequence-to-Sequence (Seq2Seq)** model, is widely used for Natural Language Processing (NLP) tasks where both the input and output are sequences.

Examples include:

- Machine Translation
- Text Summarization
- Chatbots
- Speech Recognition

The architecture consists of two major components:

1. Encoder
2. Decoder

---

### Encoder

The encoder reads the input sentence one word at a time.

At each time step, it updates its hidden state using the current word and the previous hidden state.

Example input sentence:

```
I am a student .
```

After processing the complete sentence, the encoder generates a **context vector**, which summarizes the meaning of the entire input sentence.

---

### 4. Decoder

The decoder receives the context vector generated by the encoder and starts producing the translated sentence one word at a time.

For example,

Input:

```
I am a student .
```

Output:

```
Je suis étudiant .
```

The decoder continues generating words until the **EOS (End of Sentence)** token is produced.

---

## Architecture of Basic Encoder–Decoder

```text
             INPUT SENTENCE

      +-------+-------+-------+-------+
      |   I   |  am   |   a   |student|
      +-------+-------+-------+-------+
           |       |       |       |
           v       v       v       v

      +----------------------------------+
      |          ENCODER RNN             |
      |                                  |
      | h1 → h2 → h3 → h4                |
      +----------------------------------+
                     |
                     |
              Context Vector
                     |
                     v

      +----------------------------------+
      |          DECODER RNN             |
      |                                  |
      | <SOS> → Je → suis → étudiant →EOS|
      +----------------------------------+

In [1]:
# requirements
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


In [2]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [3]:
# Turn a Unicode string to plain ASCII, thanks to
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [4]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

In [5]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [6]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [7]:
PATH = r'/content/data.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words.

Reading lines...
Read 106112 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['vous etes trop polie', 'you re too polite']


15

In [8]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

In [9]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1) # values, index of the highest-scoring word
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input (removes the last dimension before detaching)

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)
        output = self.out(output)
        return output, hidden

In [10]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [11]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [12]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [13]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [14]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [15]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [16]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

### Training and Evaluating

In [17]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 127616 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 15s (- 10m 3s) (5 2%) 1.9643
0m 23s (- 7m 34s) (10 5%) 1.2874
0m 33s (- 6m 51s) (15 7%) 1.1035
0m 42s (- 6m 24s) (20 10%) 0.9839
0m 51s (- 6m 3s) (25 12%) 0.8818
1m 0s (- 5m 44s) (30 15%) 0.7980
1m 10s (- 5m 31s) (35 17%) 0.7282
1m 20s (- 5m 21s) (40 20%) 0.6651
1m 28s (- 5m 6s) (45 22%) 0.6061
1m 38s (- 4m 54s) (50 25%) 0.5515
1m 47s (- 4m 44s) (55 27%) 0.4999
1m 57s (- 4m 33s) (60 30%) 0.4528
2m 5s (- 4m 21s) (65 32%) 0.4124
2m 15s (- 4m 10s) (70 35%) 0.3701
2m 24s (- 4m 0s) (75 37%) 0.3376
2m 33s (- 3m 49s) (80 40%) 0.3057
2m 42s (- 3m 39s) (85 42%) 0.2781
2m 51s (- 3m 30s) (90 45%) 0.2527
3m 1s (- 3m 20s) (95 47%) 0.2279
3m 9s (- 3m 9s) (100 50%) 0.2077
3m 19s (- 3m 0s) (105 52%) 0.1908
3m 28s (- 2m 50s) (110 55%) 0.1763
3m 37s (- 2m 40s) (115 57%) 0.1600
3m 47s (- 2m 31s) (120 60%) 0.1468
3m 56s (- 2m 21s) (125 62%) 0.1360
4m 6s (- 2m 12s) (130 65%) 0.126

In [18]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> nous sommes prudentes
= we re careful
< you re the greatest <EOS>

> nous sommes puissants
= we re powerful
< you re in danger <EOS>

> vous etes paresseux
= you re lazy
< i m from kyoto <EOS>

> il est ruine
= he s broke
< you re all happy <EOS>

> j en suis sure
= i m sure
< i m sure <EOS>

> il est tres talentueux
= he s very talented
< he s very talented <EOS>

> il est faineant
= he is lazy
< you re so funny <EOS>

> vous etes tres effronte
= you re very forward
< you re very forward <EOS>

> j ai toujours faim
= i m always hungry
< i m still hungry <EOS>

> je suis ton amie
= i m your friend
< i m your friend <EOS>



## Discussion
This lab successfully demonstrated an RNN Encoder-Decoder model for French-to-English translation. The model learned to translate short, simple phrases, as evidenced by the consistent decrease in training loss from over 200 epochs, likely aided by teacher forcing. While it performed well on familiar patterns (e.g., "we are men"), its limitations were apparent in less direct or unseen phrases, often leading to irrelevant or generalized translations (e.g., "we're being attacked" translated as "i'm a salesperson"). These shortcomings stem from the simple RNN architecture, the strict MAX_LENGTH = 5 constraint, the relatively small and filtered dataset, and the absence of an attention mechanism. The model lacks deep contextual understanding, relying more on superficial patterns.

## Conclusion
The basic RNN Encoder-Decoder model effectively translates simple French-to-English phrases but struggles with complexity and novelty. Its success on learned patterns highlights the potential of sequence-to-sequence models, while its failures on divergent inputs underscore the need for more advanced architectures (e.g., LSTMs/GRUs, Attention), larger datasets, and objective evaluation metrics (like BLEU) for robust neural machine translation. This foundational labwork offers insight into the core principles and challenges of building such systems.